In [5]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches

# --- Physics & Model Parameters ---
N_ANTS = 30
F_0 = 2.0             
F_IND = 15.0          
K_C = 0.5             
GAMMA = 5.0           
GAMMA_ROT = 40.0      
F_KIN_0 = 10.0        
BETA = 0.5            
DT = 0.02             

# --- Environment constraints ---
HALL_W = 2.0

def get_wall_reaction(x, y):
    """Penalty method for wall collisions. Returns reaction force vector."""
    fx, fy = 0.0, 0.0
    stiffness = 500.0  # Increased for an impenetrable concrete wall
    
    if x < -HALL_W: fx += stiffness * (-HALL_W - x)
    if y < -HALL_W: fy += stiffness * (-HALL_W - y)
    
    if x > HALL_W and y > HALL_W:
        dx = x - HALL_W
        dy = y - HALL_W
        if dx > dy: fy -= stiffness * dy
        else:       fx -= stiffness * dx
        
    return np.array([fx, fy])

class Ant:
    def __init__(self, local_x, local_y):
        self.local_pos = np.array([local_x, local_y])
        self.is_puller = True
        self.angle = 0.0
        self.informed = False

class Piano:
    def __init__(self):
        self.pos = np.array([0.0, 15.0]) 
        self.theta = 0.0                 
        self.length = 7.0                
        self.width = 1.5
        self.target = np.array([15.0, 0.0])
        
        self.ants = []
        l, w = self.length/2, self.width/2
        perimeter = 2 * self.length + 2 * self.width
        
        # Dynamically spawn ants around the perimeter
        for _ in range(N_ANTS):
            edge_pos = np.random.uniform(0, perimeter)
            if edge_pos < self.length:
                x, y = -w, -l + edge_pos
            elif edge_pos < self.length + self.width:
                x, y = -w + (edge_pos - self.length), l
            elif edge_pos < 2*self.length + self.width:
                x, y = w, l - (edge_pos - self.length - self.width)
            else:
                x, y = w - (edge_pos - 2*self.length - self.width), -l
                
            ant = Ant(x, y)
            
            # Any ant has a 15% chance to be "informed"
            if np.random.rand() < 0.15: 
                ant.informed = True
                
            self.ants.append(ant)

    def get_corners(self):
        """For rendering the visual polygon."""
        rot_matrix = np.array([[np.cos(self.theta), -np.sin(self.theta)],
                               [np.sin(self.theta),  np.cos(self.theta)]])
        l, w = self.length/2, self.width/2
        local_corners = [np.array([-w, -l]), np.array([w, -l]), 
                         np.array([w, l]), np.array([-w, l])]
        return [self.pos + np.dot(rot_matrix, c) for c in local_corners]

    def get_dense_perimeter(self):
        """Generates a dense shield of collision points to prevent wall ghosting."""
        points = []
        l, w = self.length / 2, self.width / 2
        
        # Create a point every 0.2 units along all four edges
        for y in np.arange(-l, l + 0.2, 0.2):
            points.extend([np.array([-w, y]), np.array([w, y])])
        for x in np.arange(-w, w + 0.2, 0.2):
            points.extend([np.array([x, -l]), np.array([x, l])])
            
        rot_matrix = np.array([[np.cos(self.theta), -np.sin(self.theta)],
                               [np.sin(self.theta),  np.cos(self.theta)]])
        
        return [self.pos + np.dot(rot_matrix, p) for p in points]

    def update_physics(self):
        rot_matrix = np.array([[np.cos(self.theta), -np.sin(self.theta)],
                               [np.sin(self.theta),  np.cos(self.theta)]])
        
        f_ants = np.zeros(2)
        tau_ants = 0.0
        f_wall = np.zeros(2)
        tau_wall = 0.0

        # --- FIX: Use the dense shield of points for collision detection ---
        collision_points = self.get_dense_perimeter()
            
        for global_c in collision_points:
            reaction = get_wall_reaction(global_c[0], global_c[1])
            f_wall += reaction
            r_vec = global_c - self.pos 
            tau_wall += np.cross(r_vec, reaction)
        # -------------------------------------------------------------------

        for ant in self.ants:
            r_global = np.dot(rot_matrix, ant.local_pos)
            
            if ant.is_puller:
                if ant.informed:
                    direction = self.target - self.pos
                    norm = np.linalg.norm(direction)
                    pull_vec = (direction / norm) * F_0 if norm > 0 else np.zeros(2)
                else:
                    global_angle = self.theta + ant.angle
                    pull_vec = np.array([np.cos(global_angle), np.sin(global_angle)]) * F_0
                
                f_ants += pull_vec
                tau_ants += np.cross(r_global, pull_vec)

        num_lifters = sum(1 for a in self.ants if not a.is_puller)
        f_kin = max(F_KIN_0 - BETA * num_lifters, 0.0)
        
        f_total_applied = f_ants + f_wall
        speed_applied = np.linalg.norm(f_total_applied)
        
        if speed_applied > f_kin:
            f_net = f_total_applied - (f_total_applied / speed_applied) * f_kin
        else:
            f_net = np.zeros(2) 
            
        tau_total = tau_ants + tau_wall
        if abs(tau_total) > f_kin:
            tau_net = tau_total - np.sign(tau_total) * f_kin
        else:
            tau_net = 0.0

        v_cm = f_net / GAMMA
        omega = tau_net / GAMMA_ROT
        
        v_cm = np.clip(v_cm, -8.0, 8.0)
        omega = np.clip(omega, -2.0, 2.0)

        b_radius = max(self.length, self.width) / 2.0

        for ant in self.ants:
            if ant.informed: continue
            
            r_global = np.dot(rot_matrix, ant.local_pos)
            tangential_dir = np.array([-r_global[1], r_global[0]])
            f_rot_i = (GAMMA_ROT / b_radius) * omega * tangential_dir
            sensed_f_i = f_net - f_rot_i
            
            norm_f = np.linalg.norm(sensed_f_i)
            energy_term = 0.0
            
            if norm_f > 0:
                global_angle = self.theta + ant.angle
                p_i = np.array([np.cos(global_angle), np.sin(global_angle)])
                energy_term = np.dot(p_i, sensed_f_i) / F_IND
                
            if ant.is_puller:
                rate_to_lift = K_C * np.exp(-energy_term)
                if np.random.rand() < rate_to_lift * DT:
                    ant.is_puller = False
            else:
                rate_to_pull = K_C * np.exp(energy_term)
                if np.random.rand() < rate_to_pull * DT:
                    ant.is_puller = True
                    if norm_f > 0:
                        noise = np.random.normal(0, 0.5) 
                        ant.angle = np.arctan2(sensed_f_i[1], sensed_f_i[0]) - self.theta + noise

        self.pos += v_cm * DT
        self.theta += omega * DT
        
        if np.isnan(self.pos).any() or np.isnan(self.theta):
            self.pos = np.array([0.0, 15.0])
            self.theta = 0.0

# --- Visualization Setup ---
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(-5, 20)
ax.set_ylim(-5, 20)
ax.set_aspect('equal')
ax.set_title("Fully Emergent Swarm - Impenetrable Walls")

walls = [
    patches.Rectangle((-5, -5), 3, 25, color='gray', zorder=1),  
    patches.Rectangle((-5, -5), 25, 3, color='gray', zorder=1),  
    patches.Rectangle((2, 2), 18, 18, color='gray', zorder=1)    
]
for w in walls: ax.add_patch(w)

piano = Piano()
ax.plot(piano.target[0], piano.target[1], 'r*', markersize=15, label="Nest", zorder=2)

piano_patch = patches.Polygon(piano.get_corners(), closed=True, color='saddlebrown', alpha=0.8, zorder=3)
ax.add_patch(piano_patch)

pullers_plot, = ax.plot([], [], 'bo', markersize=4, label="Pullers", zorder=4)
lifters_plot, = ax.plot([], [], 'ro', markersize=4, label="Lifters", zorder=4)
ax.legend(loc='upper right')

def init():
    piano_patch.set_xy(piano.get_corners())
    pullers_plot.set_data([], [])
    lifters_plot.set_data([], [])
    return piano_patch, pullers_plot, lifters_plot

def animate(frame):
    for _ in range(5): 
        piano.update_physics()
    
    piano_patch.set_xy(piano.get_corners())
    
    rot_matrix = np.array([[np.cos(piano.theta), -np.sin(piano.theta)],
                           [np.sin(piano.theta),  np.cos(piano.theta)]])
    
    px, py, lx, ly = [], [], [], []
    for ant in piano.ants:
        global_pos = piano.pos + np.dot(rot_matrix, ant.local_pos)
        if ant.is_puller:
            px.append(global_pos[0])
            py.append(global_pos[1])
        else:
            lx.append(global_pos[0])
            ly.append(global_pos[1])
            
    pullers_plot.set_data(px, py)
    lifters_plot.set_data(lx, ly)
    
    return piano_patch, pullers_plot, lifters_plot

ani = animation.FuncAnimation(fig, animate, init_func=init, frames=800, interval=20, blit=False)
plt.show()
ani.save("ants1.gif", writer='pillow', fps=25)

D:\ANSYS_TEMP\ipykernel_29028\3548649858.py:249: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
D:\ANSYS_TEMP\ipykernel_29028\3548649858.py:116: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  tau_wall += np.cross(r_vec, reaction)
D:\ANSYS_TEMP\ipykernel_29028\3548649858.py:132: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  tau_ants += np.cross(r_global, pull_vec)


KeyboardInterrupt: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation

# ── Model parameters (from paper Supplementary Note 13) ──────────────────────
N_ANTS     = 30
F_0        = 2.0        # force per ant (ant-force units)
F_IND      = 10.0       # individuality parameter  (paper: Find = 10)
K_C        = 0.7        # conversion/reorientation rate  (paper: Kc = 0.7 /s)
GAMMA      = 25.0       # translational drag  (paper: γ = 25)
GAMMA_ROT  = 0.4 * GAMMA  # rotational drag  (paper: γ_rot ≈ 0.4·b·γ, b≈1)
F_KIN_0    = 10.0       # baseline kinetic friction
BETA       = 1.65       # lifter friction-reduction factor  (paper: β = 1.65)
K_FORGET   = 0.1        # informed-ant forgetting rate  (paper: Kforget = 0.1 /s)
DT         = 0.02       # timestep (s)
N_STEPS    = 3000       # simulation steps  (~60 s total)

# ── Load geometry ─────────────────────────────────────────────────────────────
LOAD_L = 3.5   # half-length
LOAD_W = 0.75  # half-width

SLOPE_ANGLE_DEG = 15.0   # reference slope angle (used in individual plots)


def make_ants():
    """Place N_ANTS randomly around the rectangular perimeter."""
    local_positions = []
    L, W = LOAD_L, LOAD_W
    perimeter = 2 * (2*L + 2*W)
    for _ in range(N_ANTS):
        ep = np.random.uniform(0, 2*(2*L + 2*W))
        if   ep < 2*L:           x, y = -W,  -L + ep/2
        elif ep < 2*L + 2*W:     x, y = -W + (ep - 2*L)/2,  L
        elif ep < 4*L + 2*W:     x, y =  W,  L - (ep - 2*L - 2*W)/2
        else:                    x, y =  W - (ep - 4*L - 2*W)/2, -L
        local_positions.append(np.array([x, y]))
    return local_positions


def simulate(slope_angle_deg=0.0, seed=42):
    """
    Run one simulation.

    slope_angle_deg > 0  → upslope   (gravity opposes motion)
    slope_angle_deg < 0  → downslope (gravity aids motion)
    slope_angle_deg = 0  → flat
    """
    np.random.seed(seed)

    slope = np.radians(slope_angle_deg)
    g_magnitude = F_0 * N_ANTS * 0.3 * np.abs(np.sin(slope))
    f_gravity = np.array([-np.sign(slope_angle_deg) * g_magnitude, 0.0])

    pos      = np.array([0.0, 0.0])
    theta    = 0.0
    target   = np.array([30.0, 4.0])

    local_pos = make_ants()
    is_puller = np.ones(N_ANTS, dtype=bool)
    angles    = np.zeros(N_ANTS)
    informed  = np.random.rand(N_ANTS) < 0.15

    trajectory = [pos.copy()]

    for _ in range(N_STEPS):
        rot = np.array([[np.cos(theta), -np.sin(theta)],
                        [np.sin(theta),  np.cos(theta)]])

        f_ants  = np.zeros(2)
        tau_ants = 0.0

        for i in range(N_ANTS):
            if not is_puller[i]:
                continue
            r_g = rot @ local_pos[i]
            if informed[i]:
                d = target - pos
                n = np.linalg.norm(d)
                pv = (d / n) * F_0 if n > 0 else np.zeros(2)
            else:
                ga = theta + angles[i]
                pv = np.array([np.cos(ga), np.sin(ga)]) * F_0
            f_ants  += pv
            tau_ants += float(np.cross(r_g, pv))

        f_total = f_ants + f_gravity

        n_lifters = int(np.sum(~is_puller))
        f_kin = max(I2_F_KIN_0 - I2_BETA * n_lifters * I2_F_0, 0.0)

        spd = np.linalg.norm(f_total)
        f_net = (f_total - (f_total / spd) * f_kin) if spd > f_kin else np.zeros(2)
        tau_net = (tau_ants - np.sign(tau_ants) * f_kin) \
                  if abs(tau_ants) > f_kin else 0.0

        v_cm  = np.clip(f_net / GAMMA, -5.0, 5.0)
        omega = np.clip(tau_net / GAMMA_ROT, -1.0, 1.0)
        b_radius = max(LOAD_L, LOAD_W) / 2.0

        for i in range(N_ANTS):
            if informed[i] and np.random.rand() < K_FORGET * DT:
                informed[i] = False
            if informed[i]:
                continue
            r_g   = rot @ local_pos[i]
            tang  = np.array([-r_g[1], r_g[0]])
            f_rot_i = (GAMMA_ROT / b_radius) * omega * tang
            sensed  = f_net - f_rot_i

            norm_s = np.linalg.norm(sensed)
            energy = 0.0
            if norm_s > 0:
                ga  = theta + angles[i]
                p_i = np.array([np.cos(ga), np.sin(ga)])
                energy = np.dot(p_i, sensed) / F_IND

            if is_puller[i]:
                if np.random.rand() < K_C * np.exp(-energy) * DT:
                    is_puller[i] = False
            else:
                if np.random.rand() < K_C * np.exp(energy) * DT:
                    is_puller[i] = True
                    if norm_s > 0:
                        angles[i] = (np.arctan2(sensed[1], sensed[0])
                                     - theta + np.random.normal(0, 0.52))

        pos   = pos + v_cm * DT
        theta = theta + omega * DT
        trajectory.append(pos.copy())

    return np.array(trajectory)


# ── Run three scenarios ───────────────────────────────────────────────────────
print("Simulating flat..."); traj_flat = simulate(  0.0)
print("Simulating upslope..."); traj_up = simulate( 15.0)
print("Simulating downslope..."); traj_dn = simulate(-15.0)

scenarios = [
    (traj_flat, "Flat (0°)",        "#1D9E75",  0.0),
    (traj_up,   "Upslope (15°)",    "#D85A30",  15.0),
    (traj_dn,   "Downslope (15°)", "#378ADD", -15.0),
]

# ── Plot 1: Paths projected on inclined plane (colour = time) ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, (traj, title, color, angle_deg) in zip(axes, scenarios):

    # CHANGE 3: project x onto inclined plane (foreshortening)
    slope_rad = np.radians(angle_deg)
    cos_s = np.cos(slope_rad)
    traj_display = traj.copy()
    traj_display[:, 0] = traj[:, 0] * cos_s
    nest_x_display = 20.0 * cos_s

    # Faint path line + time-coloured dots
    N = len(traj_display)
    times = np.arange(N) * DT
    ax.plot(traj_display[:, 0], traj_display[:, 1],
            color=color, lw=1.0, alpha=0.35, zorder=1)
    sc = ax.scatter(traj_display[::10, 0], traj_display[::10, 1],
                    c=times[::10], cmap="viridis", s=6, zorder=2,
                    vmin=0, vmax=N_STEPS * DT)

    ax.plot(traj_display[0, 0], traj_display[0, 1],
            "ko", ms=8, label="Start", zorder=5)
    ax.plot(nest_x_display, 0, "r*", ms=16, label="Nest", zorder=5)

    # CHANGE 4: gravity arrow
    if angle_deg != 0:
        arr_x, arr_y = -3.5, -7.5
        arrow_dx = 2.5 if angle_deg < 0 else -2.5
        ax.annotate("", xy=(arr_x + arrow_dx, arr_y), xytext=(arr_x, arr_y),
                    arrowprops=dict(arrowstyle="->", color="orange", lw=2))
        ax.text(arr_x + arrow_dx / 2, arr_y - 1.2, "gravity",
                fontsize=8, color="orange", ha="center")

    plt.colorbar(sc, ax=ax, label="Time (s)", pad=0.02)
    ax.set_xlim(-6, 22); ax.set_ylim(-12, 12)
    ax.set_aspect("equal")

    # CHANGE 2: updated axis labels and title
    ax.set_title(f"Projected path on inclined plane\n{title}", fontsize=12)
    ax.set_xlabel("Along-slope distance (cm)", fontsize=10)
    ax.set_ylabel("Across-slope distance (cm)", fontsize=10)

    ax.axhline(0, color="gray", lw=0.5, ls="--")
    ax.axvline(0, color="gray", lw=0.5, ls="--")
    ax.legend(fontsize=9, loc="upper left")
    ax.grid(True, alpha=0.25)

plt.suptitle("Ant collective transport — path projected on inclined plane\n(colour = time elapsed)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("inclined_plane_paths.png", dpi=140, bbox_inches="tight")
plt.show()

# ── Plot 2: x-displacement over time ─────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(9, 5))
for traj, label, color, angle_deg in scenarios:
    t = np.arange(len(traj)) * DT
    ax2.plot(t, traj[:, 0], label=label, color=color, lw=2)

ax2.axhline(20, color="red", ls="--", lw=1, label="Nest position")
ax2.set_xlabel("Time (s)", fontsize=12)
ax2.set_ylabel("x-displacement (cm)", fontsize=12)
ax2.set_title("Progress toward nest over time", fontsize=13)
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("displacement.png", dpi=130, bbox_inches="tight")
plt.show()

print("Done! Saved inclined_plane_paths.png and displacement.png")

Simulating flat...


D:\ANSYS_TEMP\ipykernel_29028\3097413307.py:85: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  tau_ants += float(np.cross(r_g, pv))


Simulating upslope...
Simulating downslope...
Done! Saved inclined_plane_paths.png and displacement.png


D:\ANSYS_TEMP\ipykernel_29028\3097413307.py:197: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
D:\ANSYS_TEMP\ipykernel_29028\3097413307.py:212: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation
from matplotlib.patches import Polygon

# ── Model parameters (from paper Supplementary Note 13) ──────────────────────
N_ANTS    = 30
F_0       = 2.0        # force per ant (ant-force units)
F_IND     = 10.0       # individuality parameter  (paper: Find = 10)
K_C       = 0.7        # conversion/reorientation rate  (paper: Kc = 0.7 /s)
GAMMA     = 25.0       # translational drag  (paper: γ = 25)
GAMMA_ROT = 0.4 * GAMMA  # rotational drag  (paper: γ_rot ≈ 0.4·b·γ, b≈1)
F_KIN_0   = 10.0       # baseline kinetic friction
BETA      = 1.65       # lifter friction-reduction factor  (paper: β = 1.65)
K_FORGET  = 0.1        # informed-ant forgetting rate  (paper: Kforget = 0.1 /s)
DT        = 0.02       # timestep (s)
N_STEPS   = 2000       # simulation steps (~40 s total)

# ── Load geometry ─────────────────────────────────────────────────────────────
LOAD_L = 3.5   # half-length
LOAD_W = 0.75  # half-width

# Board geometry
BOARD_XMIN, BOARD_XMAX = -10.0, 40.0
BOARD_YMIN, BOARD_YMAX = -15.0, 15.0

def make_ants(seed):
    """Place N_ANTS randomly around the rectangular perimeter."""
    np.random.seed(seed)
    lp = []
    L, W = LOAD_L, LOAD_W
    for _ in range(N_ANTS):
        ep = np.random.uniform(0, 2*(2*L + 2*W))
        if   ep < 2*L:         x, y = -W,  -L + ep/2
        elif ep < 2*L + 2*W:   x, y = -W + (ep - 2*L)/2,  L
        elif ep < 4*L + 2*W:   x, y =  W,  L - (ep - 2*L - 2*W)/2
        else:                  x, y =  W - (ep - 4*L - 2*W)/2, -L
        lp.append(np.array([x, y]))
    return lp


def simulate_full(slope_angle_deg=0.0, seed=42):
    """
    Run simulation and return full history of states.

    slope_angle_deg > 0  → upslope   (gravity opposes motion)
    slope_angle_deg < 0  → downslope (gravity aids motion)
    slope_angle_deg = 0  → flat

    Returns:
        hist_pos      : (N_STEPS+1, 2) array of load positions
        hist_theta    : (N_STEPS+1,)   array of load orientations
        hist_puller   : (N_STEPS+1, N_ANTS) bool array
        hist_informed : (N_STEPS+1, N_ANTS) bool array
        local_pos     : list of N_ANTS local attachment positions
    """
    np.random.seed(seed)
    slope = np.radians(slope_angle_deg)
    g_mag = F_0 * N_ANTS * 0.3 * abs(np.sin(slope))
    f_gravity = np.array([-np.sign(slope_angle_deg) * g_mag, 0.0])

    pos    = np.array([0.0, 0.0])
    theta  = 0.0
    target = np.array([30.0, 4.0])

    local_pos = make_ants(seed)
    is_puller = np.ones(N_ANTS, dtype=bool)
    angles    = np.zeros(N_ANTS)
    informed  = np.random.rand(N_ANTS) < 0.15   # ~15% start informed

    hist_pos      = [pos.copy()]
    hist_theta    = [theta]
    hist_puller   = [is_puller.copy()]
    hist_informed = [informed.copy()]

    for _ in range(N_STEPS):
        rot = np.array([[np.cos(theta), -np.sin(theta)],
                        [np.sin(theta),  np.cos(theta)]])

        # ── Ant forces & torques ──────────────────────────────────────────
        f_ants, tau_ants = np.zeros(2), 0.0
        for i in range(N_ANTS):
            if not is_puller[i]:
                continue
            r_g = rot @ local_pos[i]
            if informed[i]:
                d = target - pos
                n = np.linalg.norm(d)
                pv = (d / n) * F_0 if n > 0 else np.zeros(2)
            else:
                ga = theta + angles[i]
                pv = np.array([np.cos(ga), np.sin(ga)]) * F_0
            f_ants   += pv
            tau_ants += float(np.cross(r_g, pv))

        # ── Slope gravity ─────────────────────────────────────────────────
        f_total   = f_ants + f_gravity
        n_lifters = int(np.sum(~is_puller))
        f_kin     = max(F_KIN_0 - BETA * n_lifters * F_0 * 0.1, 0.0)
        spd       = np.linalg.norm(f_total)
        f_net     = (f_total - (f_total / spd) * f_kin) if spd > f_kin else np.zeros(2)
        tau_net   = (tau_ants - np.sign(tau_ants) * f_kin) \
                    if abs(tau_ants) > f_kin else 0.0

        v_cm     = np.clip(f_net / GAMMA, -5., 5.)
        omega    = np.clip(tau_net / GAMMA_ROT, -1., 1.)
        b_radius = max(LOAD_L, LOAD_W) / 2.

        # ── Update ant roles ──────────────────────────────────────────────
        for i in range(N_ANTS):
            # Informed ants forget over time
            if informed[i] and np.random.rand() < K_FORGET * DT:
                informed[i] = False
            if informed[i]:
                continue
            r_g     = rot @ local_pos[i]
            tang    = np.array([-r_g[1], r_g[0]])
            f_rot_i = (GAMMA_ROT / b_radius) * omega * tang
            sensed  = f_net - f_rot_i   # Eq. (8) from paper
            norm_s  = np.linalg.norm(sensed)
            energy  = 0.0
            if norm_s > 0:
                ga  = theta + angles[i]
                p_i = np.array([np.cos(ga), np.sin(ga)])
                energy = np.dot(p_i, sensed) / F_IND
            if is_puller[i]:
                if np.random.rand() < K_C * np.exp(-energy) * DT:
                    is_puller[i] = False
            else:
                if np.random.rand() < K_C * np.exp(energy) * DT:
                    is_puller[i] = True
                    if norm_s > 0:
                        angles[i] = (np.arctan2(sensed[1], sensed[0])
                                     - theta + np.random.normal(0, 0.52))

        # ── Integrate ─────────────────────────────────────────────────────
        pos   += v_cm * DT
        theta += omega * DT
        hist_pos.append(pos.copy())
        hist_theta.append(theta)
        hist_puller.append(is_puller.copy())
        hist_informed.append(informed.copy())

    return (np.array(hist_pos), np.array(hist_theta),
            np.array(hist_puller), np.array(hist_informed),
            local_pos)


def make_gif(slope_angle_deg, filename, title):
    """Simulate and save an animated gif for a given slope angle."""
    print(f"Simulating {title}...")
    hist_pos, hist_theta, hist_puller, hist_informed, local_pos = \
        simulate_full(slope_angle_deg, seed=42)

    # x-axis foreshortening for inclined plane projection
    cos_s = np.cos(np.radians(slope_angle_deg))

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_xlim(-5, 25)
    ax.set_ylim(-12, 12)
    ax.set_aspect('equal')
    ax.set_facecolor('white')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Along-slope distance (cm)", fontsize=10)
    ax.set_ylabel("Across-slope distance (cm)", fontsize=10)
    ax.grid(True, alpha=0.25)
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.axvline(0, color='gray', lw=0.5, ls='--')

    # Nest marker
    nest_x = 20.0 * cos_s
    ax.plot(nest_x, 0, 'r*', markersize=18, zorder=6, label='Nest')

    # Gravity arrow for inclined scenarios
    if slope_angle_deg != 0:
        arr_x = -3.5
        dx = 2.5 if slope_angle_deg < 0 else -2.5
        ax.annotate('', xy=(arr_x + dx, -9), xytext=(arr_x, -9),
                    arrowprops=dict(arrowstyle='->', color='orange', lw=2))
        ax.text(arr_x + dx / 2, -10.2, 'gravity',
                fontsize=8, color='orange', ha='center')

    # Trajectory trail
    trail_line, = ax.plot([], [], color='#aaaaaa', lw=1.0, alpha=0.6, zorder=1)
    trail_x, trail_y = [], []

    # Load patch (brown rectangle)
    load_patch = Polygon(
        [[p[0] * cos_s, p[1]] for p in
         [hist_pos[0] + np.array(c) for c in
          [[-LOAD_W, -LOAD_L], [LOAD_W, -LOAD_L],
           [LOAD_W,  LOAD_L],  [-LOAD_W, LOAD_L]]]],
        closed=True, color='saddlebrown', alpha=0.85, zorder=3)
    ax.add_patch(load_patch)

    # Ant scatter plots
    pullers_plot,  = ax.plot([], [], 'bo', ms=5, label='Pullers',  zorder=5)
    lifters_plot,  = ax.plot([], [], 'ro', ms=5, label='Lifters',  zorder=5)
    informed_plot, = ax.plot([], [], 'g^', ms=6, label='Informed', zorder=5)

    time_text = ax.text(0.02, 0.96, '', transform=ax.transAxes,
                        fontsize=10, va='top')
    ax.legend(loc='upper right', fontsize=9)

    STRIDE = 5   # advance this many simulation steps per animation frame

    def init():
        trail_line.set_data([], [])
        pullers_plot.set_data([], [])
        lifters_plot.set_data([], [])
        informed_plot.set_data([], [])
        time_text.set_text('')
        return (load_patch, trail_line,
                pullers_plot, lifters_plot, informed_plot, time_text)

    def animate(frame):
        step  = frame * STRIDE
        pos   = hist_pos[step]
        theta = hist_theta[step]
        ppos  = pos * np.array([cos_s, 1.0])   # projected load centre

        # Update trail
        trail_x.append(ppos[0])
        trail_y.append(ppos[1])
        trail_line.set_data(trail_x, trail_y)

        # Update load corners (rotated + projected)
        rot = np.array([[np.cos(theta), -np.sin(theta)],
                        [np.sin(theta),  np.cos(theta)]])
        corners = []
        for cl in [np.array([-LOAD_W, -LOAD_L]), np.array([LOAD_W, -LOAD_L]),
                   np.array([LOAD_W,   LOAD_L]), np.array([-LOAD_W, LOAD_L])]:
            gp = pos + rot @ cl
            corners.append(gp * np.array([cos_s, 1.0]))
        load_patch.set_xy(corners)

        # Update ant positions
        px, py, lx, ly, ix, iy = [], [], [], [], [], []
        puller_mask   = hist_puller[step]
        informed_mask = hist_informed[step]
        for i, lp in enumerate(local_pos):
            gp = pos + rot @ lp
            dp = gp * np.array([cos_s, 1.0])
            if informed_mask[i]:
                ix.append(dp[0]); iy.append(dp[1])
            elif puller_mask[i]:
                px.append(dp[0]); py.append(dp[1])
            else:
                lx.append(dp[0]); ly.append(dp[1])

        pullers_plot.set_data(px, py)
        lifters_plot.set_data(lx, ly)
        informed_plot.set_data(ix, iy)
        time_text.set_text(f't = {step * DT:.1f} s')

        return (load_patch, trail_line,
                pullers_plot, lifters_plot, informed_plot, time_text)

    n_frames = N_STEPS // STRIDE
    ani = animation.FuncAnimation(fig, animate, init_func=init,
                                  frames=n_frames, interval=40, blit=True)
    ani.save(filename, writer='pillow', fps=25)
    plt.close()
    print(f"  saved {filename}")


# ── Generate all three gifs ───────────────────────────────────────────────────
make_gif(  0.0, 'ants_flat.gif',      'Flat (0°) — path on plane')
make_gif( 15.0, 'ants_upslope.gif',   'Upslope (15°) — path on inclined plane')
make_gif(-15.0, 'ants_downslope.gif', 'Downslope (15°) — path on inclined plane')
print("All done!")


KeyboardInterrupt: 

: 

In [ ]:


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation
from matplotlib.patches import Polygon

# SHARED GEOMETRY — I-BEAM & CORRIDOR

FLANGE_W  = 5.0   # half-width of top/bottom flange
FLANGE_H  = 0.35  # half-height of each flange
WEB_H     = 1.8   # half-height of web (gap between flanges)
WEB_W     = 0.25  # half-width of web

SLIT1_X = 6.0
SLIT2_X = 12.0
GAP_HALF = 1.2
PLATE_THICKNESS = 0.3
PLATE_HEIGHT = 10

TARGET_RADIUS = 1.0
# Narrow corridor geometry (the constriction the load must pass through)
CORRIDOR_X_START = 6.0    # where corridor begins
CORRIDOR_X_END   = 12.0   # where corridor ends
CORRIDOR_HALF_W  = 1.6    # half-width of the opening (must be > FLANGE_H but < FLANGE_W)
# NOTE: FLANGE_W (2.5) > CORRIDOR_HALF_W (1.6), so the load CANNOT pass
# in its default orientation — ants must rotate it ~90 degrees first!

CORRIDOR_WALL_THICKNESS = 3.0

# Board geometry
BOARD_XMIN, BOARD_XMAX = -10.0, 40.0
BOARD_YMIN, BOARD_YMAX = -15.0, 15.0

def ibeam_local_points(n_points=80):
    """
    Generate n_points uniformly distributed around the I-beam perimeter
    in local (body) coordinates. Used both for ant placement and
    collision detection.

    I-beam cross-section (viewed end-on):
        top flange:    y in [WEB_H, WEB_H+2*FLANGE_H], x in [-FLANGE_W, FLANGE_W]
        web:           y in [-WEB_H, WEB_H],            x in [-WEB_W,   WEB_W]
        bottom flange: y in [-(WEB_H+2*FLANGE_H), -WEB_H], x in [-FLANGE_W, FLANGE_W]

    The load travels along the x-axis (from x=0 toward x=+large).
    The long axis of the beam is along y in the default orientation.
    """
    points = []

    yT_bot = WEB_H
    yT_top = WEB_H + 2 * FLANGE_H
    yB_top = -WEB_H
    yB_bot = -(WEB_H + 2 * FLANGE_H)

    # Top flange (4 edges)
    for y in np.linspace(yT_bot, yT_top, max(4, n_points // 12)):
        points += [np.array([-FLANGE_W, y]), np.array([FLANGE_W, y])]
    for x in np.linspace(-FLANGE_W, FLANGE_W, max(8, n_points // 6)):
        points += [np.array([x, yT_top]), np.array([x, yT_bot])]

    # Web (4 edges)
    for y in np.linspace(yB_top, yT_bot, max(4, n_points // 8)):
        points += [np.array([-WEB_W, y]), np.array([WEB_W, y])]

    # Bottom flange (4 edges)
    for y in np.linspace(yB_bot, yB_top, max(4, n_points // 12)):
        points += [np.array([-FLANGE_W, y]), np.array([FLANGE_W, y])]
    for x in np.linspace(-FLANGE_W, FLANGE_W, max(8, n_points // 6)):
        points += [np.array([x, yB_bot]), np.array([x, yB_top])]

    return points
def ibeam_perimeter_sample(n_ants):
    """
    Sample n_ants positions uniformly by arc-length around the I-beam perimeter.
    Returns list of 2D numpy arrays in local coordinates.
    """
    # Build a dense outline and sample it
    outline = []
    yT_bot = WEB_H;  yT_top = WEB_H + 2 * FLANGE_H
    yB_top = -WEB_H; yB_bot = -(WEB_H + 2 * FLANGE_H)

    # Walk the outline clockwise: top-flange top → right side → ... → left side
    def linspace_pts(x0, y0, x1, y1, n=40):
        return [np.array([x0 + (x1-x0)*t, y0 + (y1-y0)*t]) for t in np.linspace(0, 1, n, endpoint=False)]

    outline += linspace_pts(-FLANGE_W, yT_top,  FLANGE_W, yT_top)   # top edge of top flange
    outline += linspace_pts( FLANGE_W, yT_top,  FLANGE_W, yT_bot)   # right edge of top flange
    outline += linspace_pts( FLANGE_W, yT_bot,   WEB_W,   yT_bot)   # step inward at web top-right
    outline += linspace_pts(  WEB_W,   yT_bot,   WEB_W,   yB_top)   # right edge of web
    outline += linspace_pts(  WEB_W,   yB_top,  FLANGE_W, yB_top)   # step outward at web bot-right
    outline += linspace_pts( FLANGE_W, yB_top,  FLANGE_W, yB_bot)   # right edge of bottom flange
    outline += linspace_pts( FLANGE_W, yB_bot, -FLANGE_W, yB_bot)   # bottom edge
    outline += linspace_pts(-FLANGE_W, yB_bot, -FLANGE_W, yB_top)   # left edge of bottom flange
    outline += linspace_pts(-FLANGE_W, yB_top,  -WEB_W,   yB_top)   # step inward at web bot-left
    outline += linspace_pts( -WEB_W,   yB_top,  -WEB_W,   yT_bot)   # left edge of web
    outline += linspace_pts( -WEB_W,   yT_bot, -FLANGE_W, yT_bot)   # step outward at web top-left
    outline += linspace_pts(-FLANGE_W, yT_bot, -FLANGE_W, yT_top)   # left edge of top flange

    # Compute cumulative arc length
    pts = np.array(outline)
    diffs = np.diff(pts, axis=0)
    seg_lens = np.linalg.norm(diffs, axis=1)
    cum_len = np.concatenate([[0], np.cumsum(seg_lens)])
    total_len = cum_len[-1]

    # Sample evenly
    sample_lens = np.linspace(0, total_len, n_ants, endpoint=False)
    sampled = []
    for sl in sample_lens:
        idx = np.searchsorted(cum_len, sl, side='right') - 1
        idx = min(idx, len(pts) - 2)
        t = (sl - cum_len[idx]) / (seg_lens[idx] + 1e-12)
        p = pts[idx] + t * (pts[idx+1] - pts[idx])
        sampled.append(p)
    return sampled

def ibeam_perimeter_sample_biased(n_ants, bias_toward_leading=True):
    """
    Sample with front-heavy bias: ~60% of ants in the leading hemisphere (x > 0 side),
    matching the paper's observation that pullers concentrate at the leading edge.
    """
    all_pts = ibeam_perimeter_sample(n_ants * 4)  # oversample
    pts_arr = np.array(all_pts)
    
    front_mask = pts_arr[:, 0] > 0   # leading edge (positive x in local frame)
    front_pts  = pts_arr[front_mask]
    back_pts   = pts_arr[~front_mask]
    
    n_front = int(n_ants * 0.60)
    n_back  = n_ants - n_front
    
    front_idx = np.random.choice(len(front_pts), n_front, replace=False)
    back_idx  = np.random.choice(len(back_pts),  n_back,  replace=False)
    
    selected = np.vstack([front_pts[front_idx], back_pts[back_idx]])
    np.random.shuffle(selected)
    return [selected[i] for i in range(n_ants)]


def ibeam_polygon_corners():
    """
    Return the I-beam polygon vertices (for matplotlib Polygon patch).
    """
    yT_bot = WEB_H;  yT_top = WEB_H + 2 * FLANGE_H
    yB_top = -WEB_H; yB_bot = -(WEB_H + 2 * FLANGE_H)
    return np.array([
        [-FLANGE_W, yT_top],
        [ FLANGE_W, yT_top],
        [ FLANGE_W, yT_bot],
        [ WEB_W,    yT_bot],
        [ WEB_W,    yB_top],
        [ FLANGE_W, yB_top],
        [ FLANGE_W, yB_bot],
        [-FLANGE_W, yB_bot],
        [-FLANGE_W, yB_top],
        [-WEB_W,    yB_top],
        [-WEB_W,    yT_bot],
        [-FLANGE_W, yT_bot],
    ])


def corridor_wall_force(global_pt):
    stiffness = 1200.0
    x, y = global_pt
    fx, fy = 0.0, 0.0

    # Board boundaries
    if x < BOARD_XMIN: fx += stiffness * (BOARD_XMIN - x)
    if x > BOARD_XMAX: fx -= stiffness * (x - BOARD_XMAX)
    if y < BOARD_YMIN: fy += stiffness * (BOARD_YMIN - y)
    if y > BOARD_YMAX: fy -= stiffness * (y - BOARD_YMAX)

    # ───────── SLIT 1 ─────────
    if SLIT1_X - PLATE_THICKNESS <= x <= SLIT1_X + PLATE_THICKNESS:
        if abs(y) > GAP_HALF:
            fx += stiffness * np.sign(SLIT1_X - x)

    # ───────── MIDDLE CHAMBER ─────────
    if SLIT1_X + PLATE_THICKNESS < x < SLIT2_X - PLATE_THICKNESS:
        # Slight vertical confinement to encourage alignment
        if y > 3.0:
            fy -= stiffness * (y - 3.0)
        if y < -3.0:
            fy += stiffness * (-3.0 - y)

    # ───────── SLIT 2 (OFFSET for angled exit) ─────────
    OFFSET = 0.8   # CRUCIAL: forces slanted exit

    if SLIT2_X - PLATE_THICKNESS <= x <= SLIT2_X + PLATE_THICKNESS:
        if abs(y - OFFSET) > GAP_HALF:
            fx += stiffness * np.sign(SLIT2_X - x)

    return np.array([fx, fy])

def slit_force(x, y, slit_x):
    stiffness = 800.0
    fx = 0.0

    # If near plate x-location
    if abs(x - slit_x) < PLATE_THICKNESS:
        # Block everywhere except gap
        if abs(y) > GAP_HALF:
            fx += stiffness * np.sign(slit_x - x)

    return fx



def get_corridor_patches():
    patches_list = []
    wall_color = '#555555'
    alpha = 0.7

    for slit_x in [SLIT1_X, SLIT2_X]:
        # top plate
        patches_list.append(patches.Rectangle(
            (slit_x, GAP_HALF),
            PLATE_THICKNESS,
            PLATE_HEIGHT,
            color=wall_color, alpha=alpha
        ))

        # bottom plate
        patches_list.append(patches.Rectangle(
            (slit_x, -PLATE_HEIGHT - GAP_HALF),
            PLATE_THICKNESS,
            PLATE_HEIGHT,
            color=wall_color, alpha=alpha
        ))

    return patches_list


# ─────────────────────────────────────────────────────────────
# IMPLEMENTATION 1 — "Piano/emergent" style (from your notebook cell 1)
# Penalty-wall forces, simple OOP, no Gillespie
# ─────────────────────────────────────────────────────────────

# --- Parameters ---
I1_N_ANTS    = 35
I1_F_0       = 3.5      
I1_F_IND     = 15.0
I1_K_C       = 0.5
I1_GAMMA     = 5.0
I1_GAMMA_ROT = 40.0
I1_F_KIN_0   = 6.0      
I1_BETA      = 0.8      # was 0.5
I1_DT        = 0.02


class AntImpl1:
    def __init__(self, local_pos):
        self.local_pos = np.array(local_pos, dtype=float)
        self.is_puller = True
        self.angle     = np.random.uniform(-np.pi, np.pi)
        self.informed  = np.random.rand() < 0.15


class LoadImpl1:
    """I-beam load for Implementation 1."""

    def __init__(self, start_pos, target):
        self.pos    = np.array(start_pos, dtype=float)
        self.theta  = 0.0          # default: long axis along y
        self.target = np.array(target, dtype=float)

        # Place ants on I-beam perimeter
        local_pts = ibeam_perimeter_sample_biased(I1_N_ANTS)
        self.ants = [AntImpl1(p) for p in local_pts]

        # Dense collision points
        self._collision_pts_local = ibeam_local_points(n_points=120)

    def _rot(self):
        c, s = np.cos(self.theta), np.sin(self.theta)
        return np.array([[c, -s], [s, c]])

    def get_polygon(self):
        R = self._rot()
        corners = ibeam_polygon_corners()
        return np.array([self.pos + R @ c for c in corners])

    def reached_target(self):
        return np.linalg.norm(self.pos - self.target) < TARGET_RADIUS

    def update(self):
        R = self._rot()

        # ── Wall reactions ────────────────────────────────────────────────
        f_wall  = np.zeros(2)
        tau_wall = 0.0
        for lp in self._collision_pts_local:
            gp = self.pos + R @ lp
            reaction = corridor_wall_force(gp)
            f_wall   += reaction
            r_vec     = gp - self.pos
            tau_wall += np.cross(r_vec, reaction)

        # ── Ant forces ────────────────────────────────────────────────────
        f_ants  = np.zeros(2)
        tau_ants = 0.0
        for ant in self.ants:
            if not ant.is_puller:
                continue
            r_g = R @ ant.local_pos
            if ant.informed:
                d = self.target - self.pos
                n = np.linalg.norm(d)
                pv = (d / n) * I1_F_0 if n > 0 else np.zeros(2)
            else:
                ga = self.theta + ant.angle
                pv = np.array([np.cos(ga), np.sin(ga)]) * I1_F_0
            f_ants   += pv
            tau_ants += float(np.cross(r_g, pv))


        # ── Net force & friction ──────────────────────────────────────────
        n_lifters = sum(1 for a in self.ants if not a.is_puller)
        f_kin = max(I1_F_KIN_0 - I1_BETA * n_lifters, 0.0)

        f_total = f_ants + f_wall
        spd = np.linalg.norm(f_total)
        f_net = (f_total - (f_total / spd) * f_kin) if spd > f_kin else np.zeros(2)

        tau_total = tau_ants + tau_wall
        tau_net = (tau_total - np.sign(tau_total) * f_kin
                   ) if abs(tau_total) > f_kin else 0.0

        v_cm  = np.clip(f_net / I1_GAMMA, -6.0, 6.0)
        omega = np.clip(tau_net / I1_GAMMA_ROT, -2.0, 2.0)

        b_radius = FLANGE_W

        # ── Update ant roles ──────────────────────────────────────────────
        for ant in self.ants:
            if ant.informed:
                continue
            r_g     = R @ ant.local_pos
            tang    = np.array([-r_g[1], r_g[0]])
            f_rot_i = (I1_GAMMA_ROT / b_radius) * omega * tang
            sensed  = f_net - f_rot_i
            norm_s  = np.linalg.norm(sensed)
            energy  = 0.0
            if norm_s > 0:
                ga  = self.theta + ant.angle
                p_i = np.array([np.cos(ga), np.sin(ga)])
                energy = np.dot(p_i, sensed) / I1_F_IND

            if ant.is_puller:
                if np.random.rand() < I1_K_C * np.exp(-energy) * I1_DT:
                    ant.is_puller = False
            else:
                if np.random.rand() < I1_K_C * np.exp(energy) * I1_DT:
                    ant.is_puller = True
                    if norm_s > 0:
                        ant.angle = (np.arctan2(sensed[1], sensed[0])- self.theta + np.random.normal(0, np.radians(52)))

        self.pos   += v_cm * I1_DT
        self.theta += omega * I1_DT

        if np.any(np.isnan(self.pos)) or np.isnan(self.theta):
            self.pos   = np.array([0.0, 0.0], dtype=float)
            self.theta = 0.0


def run_impl1_animation(save_gif=True, filename='impl1_ibeam_corridor.gif',
                        n_frames=600, substeps=5):
    """
    Run Implementation 1 and produce an animation.
    Start pos: just left of corridor. Target: right side of board.
    """
    load = LoadImpl1(start_pos=[2.0, 0.0], target=[20.0, 0.0])

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.set_xlim(-3, 22)
    ax.set_ylim(-9, 9)
    ax.set_aspect('equal')
    ax.set_facecolor('#f5f0e8')
    ax.set_title("Implementation 1 — I-beam through narrow corridor\n"
                 "(penalty-wall, emergent steering)", fontsize=12)
    ax.set_xlabel("x (cm)"); ax.set_ylabel("y (cm)")
    ax.grid(True, alpha=0.2)

    # Draw corridor walls
    for wp in get_corridor_patches():
        ax.add_patch(wp)

    # Draw corridor opening guides
    ax.axhline( CORRIDOR_HALF_W, color='red', lw=0.7, ls='--', alpha=0.5)
    ax.axhline(-CORRIDOR_HALF_W, color='red', lw=0.7, ls='--', alpha=0.5)
    ax.plot(20.0, 0.0, 'r*', ms=18, zorder=6, label='Nest / target')

    # Annotate corridor width vs beam width
    ax.annotate('', xy=(9.0,  CORRIDOR_HALF_W), xytext=(9.0, -CORRIDOR_HALF_W),
                arrowprops=dict(arrowstyle='<->', color='darkred', lw=1.5))
    ax.text(9.3, 0, f'gap={2*CORRIDOR_HALF_W:.1f}cm\nflange={2*FLANGE_W:.1f}cm',
            fontsize=7.5, color='darkred', va='center')

    load_patch = Polygon(load.get_polygon(), closed=True,
                         color='saddlebrown', alpha=0.85, zorder=4)
    ax.add_patch(load_patch)

    pullers_plt,  = ax.plot([], [], 'bo', ms=4, label='Pullers',  zorder=5)
    lifters_plt,  = ax.plot([], [], 'rs', ms=4, label='Lifters',  zorder=5)
    informed_plt, = ax.plot([], [], 'g^', ms=6, label='Informed', zorder=5)
    trail_line,   = ax.plot([], [], '-', color='#888888', lw=1, alpha=0.5, zorder=1)
    time_txt = ax.text(0.02, 0.96, '', transform=ax.transAxes, fontsize=10, va='top')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xlim(BOARD_XMIN, BOARD_XMAX)
    ax.set_ylim(BOARD_YMIN, BOARD_YMAX)
    trail_x, trail_y = [], []

    def init():
        load_patch.set_xy(load.get_polygon())
        pullers_plt.set_data([], [])
        lifters_plt.set_data([], [])
        informed_plt.set_data([], [])
        trail_line.set_data([], [])
        time_txt.set_text('')
        return load_patch, pullers_plt, lifters_plt, informed_plt, trail_line, time_txt

    def animate(frame):
        for _ in range(substeps):
            load.update()

        load_patch.set_xy(load.get_polygon())
        trail_x.append(load.pos[0]); trail_y.append(load.pos[1])
        trail_line.set_data(trail_x, trail_y)

        R = load._rot()
        px, py, lx, ly, ix, iy = [], [], [], [], [], []
        for ant in load.ants:
            gp = load.pos + R @ ant.local_pos
            if ant.informed:
                ix.append(gp[0]); iy.append(gp[1])
            elif ant.is_puller:
                px.append(gp[0]); py.append(gp[1])
            else:
                lx.append(gp[0]); ly.append(gp[1])

        pullers_plt.set_data(px, py)
        lifters_plt.set_data(lx, ly)
        informed_plt.set_data(ix, iy)
        time_txt.set_text(f't = {frame * substeps * I1_DT:.1f} s')

        if load.reached_target():
            print(f"Reached target at t = {frame * substeps * I1_DT:.2f}s")
            ani.event_source.stop()
        return load_patch, pullers_plt, lifters_plt, informed_plt, trail_line, time_txt

    ani = animation.FuncAnimation(fig, animate, init_func=init,
                                  frames=n_frames, interval=30, blit=True)
    plt.tight_layout()
    if save_gif:
        ani.save(filename, writer='pillow', fps=25)
        print(f"Saved {filename}")
    else:
        plt.show()
    plt.close()
    return ani


# ─────────────────────────────────────────────────────────────
# IMPLEMENTATION 2 — Paper's exact model (Gillespie-inspired)
# Eqs. (4,5,7,8,9), informed-ant forgetting, array-based
# ─────────────────────────────────────────────────────────────

# --- Parameters (Supplementary Note 13) ---
I2_N_ANTS    = 35
I2_F_0       = 2.0      # keep paper value
I2_F_IND     = 10.0     # paper: Find = 10
I2_K_C       = 0.7      # paper: Kc = 0.7
I2_GAMMA     = 25.0     # paper value
I2_GAMMA_ROT = 0.4 * 25.0
I2_F_KIN_0   = 4.0      # was 10.0 — paper's fstat=2.7*f0, fkin≈0.9*fstat → ~4.9
I2_BETA      = 1.65     # paper value
I2_K_FORGET  = 0.1
I2_DT        = 0.02


def simulate_impl2_with_history(n_steps=3000, seed=42):
    """
    Full paper-model simulation (Impl 2) with I-beam + corridor.
    Returns full history for animation + analysis.
    """
    np.random.seed(seed)

    pos      = np.array([2.0, 0.0], dtype=float)
    theta    = 0.0
    target   = np.array([40.0, 8.0], dtype=float)

    local_pos  = ibeam_perimeter_sample(I2_N_ANTS)
    local_pos_arr = np.array(local_pos)           # (N, 2)
    is_puller  = np.ones(I2_N_ANTS, dtype=bool)
    angles     = np.random.uniform(-np.pi, np.pi, I2_N_ANTS)
    informed = np.zeros(I2_N_ANTS, dtype=bool)
    informed[0] = True   # single informed ant at leading-edge position
    
    collision_pts_local = np.array(ibeam_local_points(n_points=120))

    hist_pos      = [pos.copy()]
    hist_theta    = [theta]
    hist_puller   = [is_puller.copy()]
    hist_informed = [informed.copy()]

    for _ in range(n_steps):
        c, s = np.cos(theta), np.sin(theta)
        R = np.array([[c, -s], [s, c]])

        # ── Wall reactions ────────────────────────────────────────────────
        f_wall   = np.zeros(2)
        tau_wall = 0.0
        for lp in collision_pts_local:
            gp       = pos + R @ lp
            reaction = corridor_wall_force(gp)
            f_wall  += reaction
            r_vec    = gp - pos
            tau_wall += float(np.cross(r_vec, reaction))

        # ── Ant forces & torques (Eqs. 4–5) ──────────────────────────────
        f_ants   = np.zeros(2)
        tau_ants = 0.0
        for i in range(I2_N_ANTS):
            if not is_puller[i]:
                continue
            r_g = R @ local_pos_arr[i]
            if informed[i]:
                d = target - pos
                n = np.linalg.norm(d)
                pv = (d / n) * I2_F_0 if n > 0 else np.zeros(2)
            else:
                ga = theta + angles[i]
                pv = np.array([np.cos(ga), np.sin(ga)]) * I2_F_0
            f_ants   += pv
            tau_ants += float(np.cross(r_g, pv))

        # ── Friction & net force ──────────────────────────────────────────
        n_lifters = int(np.sum(~is_puller))
        # Eq.(1) style: lifters reduce friction by β·f0 each
        f_kin   = max(I2_F_KIN_0 - I2_BETA * n_lifters * I2_F_0 * 0.1, 0.0)

        f_total  = f_ants + f_wall
        spd      = np.linalg.norm(f_total)
        f_net    = (f_total - (f_total / spd) * f_kin) if spd > f_kin else np.zeros(2)
        tau_net  = (tau_ants + tau_wall - np.sign(tau_ants + tau_wall) * f_kin
                    ) if abs(tau_ants + tau_wall) > f_kin else 0.0

        v_cm  = np.clip(f_net / I2_GAMMA, -5.0, 5.0)
        omega = np.clip(tau_net / I2_GAMMA_ROT, -1.2, 1.2)
        b_radius = FLANGE_W

        # ── Update ant roles (Eqs. 7–9 from paper) ───────────────────────
        for i in range(I2_N_ANTS):
            # Informed ants forget with rate K_FORGET
            if informed[i] and np.random.rand() < I2_K_FORGET * I2_DT:
                informed[i] = False
            if informed[i]:
                continue
            if np.random.rand() < 0.03 * I2_DT:   # Kknow = 0.03/s
                candidate = np.random.randint(I2_N_ANTS)
                informed[candidate] = True

            r_g     = R @ local_pos_arr[i]
            tang    = np.array([-r_g[1], r_g[0]])
            # Eq.(8): ant senses translational force minus rotational component
            f_rot_i = (I2_GAMMA_ROT / b_radius) * omega * tang
            sensed  = f_net - f_rot_i
            norm_s  = np.linalg.norm(sensed)
            energy  = 0.0
            if norm_s > 0:
                ga  = theta + angles[i]
                p_i = np.array([np.cos(ga), np.sin(ga)])
                energy = np.dot(p_i, sensed) / I2_F_IND  # Eq.(7) exponent

            # Eq.(9): role-switching rates
            if is_puller[i]:
                rate = I2_K_C * np.exp(-energy)
                if np.random.rand() < rate * I2_DT:
                    is_puller[i] = False
            else:
                rate = I2_K_C * np.exp(energy)
                if np.random.rand() < rate * I2_DT:
                    is_puller[i] = True
                    if norm_s > 0:
                        # Newly-pulling ant aligns with sensed force + noise (σ=52°)
                        angles[i] = (np.arctan2(sensed[1], sensed[0])- theta + np.random.normal(0, np.radians(52)))

        # Integrate 
        pos   += v_cm * I2_DT
        theta += omega * I2_DT

        hist_pos.append(pos.copy())
        hist_theta.append(theta)
        hist_puller.append(is_puller.copy())
        hist_informed.append(informed.copy())

    return (np.array(hist_pos), np.array(hist_theta),
            np.array(hist_puller), np.array(hist_informed),
            local_pos_arr)


def run_impl2_animation(save_gif=True, filename='impl2_ibeam_corridor.gif',
                        n_steps=3000, stride=5):
    """Run Implementation 2 and produce an animation."""
    print("Running Implementation 2 simulation...")
    hist_pos, hist_theta, hist_puller, hist_informed, local_pos_arr = \
        simulate_impl2_with_history(n_steps=n_steps)
    target = np.array([30.0, 8.0])
    n_frames = n_steps // stride

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.set_xlim(-3, 22)
    ax.set_ylim(-9, 9)
    ax.set_aspect('equal')
    ax.set_facecolor('#f5f0e8')
    ax.set_title("Implementation 2 — I-beam through narrow corridor\n"
                 "(paper's Eqs.7–9, Gillespie-inspired, informed-ant forgetting)", fontsize=12)
    ax.set_xlabel("x (cm)"); ax.set_ylabel("y (cm)")
    ax.grid(True, alpha=0.2)

    for wp in get_corridor_patches():
        ax.add_patch(wp)

    ax.axhline( CORRIDOR_HALF_W, color='red', lw=0.7, ls='--', alpha=0.5)
    ax.axhline(-CORRIDOR_HALF_W, color='red', lw=0.7, ls='--', alpha=0.5)
    ax.plot(20.0, 0.0, 'r*', ms=18, zorder=6, label='Nest / target')
    ax.annotate('', xy=(9.0,  CORRIDOR_HALF_W), xytext=(9.0, -CORRIDOR_HALF_W),
                arrowprops=dict(arrowstyle='<->', color='darkred', lw=1.5))
    ax.text(9.3, 0, f'gap={2*CORRIDOR_HALF_W:.1f}cm\nflange={2*FLANGE_W:.1f}cm',
            fontsize=7.5, color='darkred', va='center')

    # Initial polygon
    R0 = np.array([[np.cos(hist_theta[0]), -np.sin(hist_theta[0])],
                   [np.sin(hist_theta[0]),  np.cos(hist_theta[0])]])
    init_corners = np.array([hist_pos[0] + R0 @ c for c in ibeam_polygon_corners()])
    load_patch = Polygon(init_corners, closed=True,
                         color='steelblue', alpha=0.85, zorder=4)
    ax.add_patch(load_patch)

    pullers_plt,  = ax.plot([], [], 'bo', ms=4, label='Pullers',  zorder=5)
    lifters_plt,  = ax.plot([], [], 'rs', ms=4, label='Lifters',  zorder=5)
    informed_plt, = ax.plot([], [], 'g^', ms=6, label='Informed', zorder=5)
    trail_line,   = ax.plot([], [], '-', color='#888888', lw=1, alpha=0.5, zorder=1)
    time_txt = ax.text(0.02, 0.96, '', transform=ax.transAxes, fontsize=10, va='top')
    ax.legend(loc='upper right', fontsize=9)

    trail_x, trail_y = [], []

    def init():
        pullers_plt.set_data([], [])
        lifters_plt.set_data([], [])
        informed_plt.set_data([], [])
        trail_line.set_data([], [])
        time_txt.set_text('')
        return load_patch, pullers_plt, lifters_plt, informed_plt, trail_line, time_txt

    def animate(frame):
        step  = frame * stride
        pos   = hist_pos[step]
        theta = hist_theta[step]
        c, s  = np.cos(theta), np.sin(theta)
        R     = np.array([[c, -s], [s, c]])

        corners = np.array([pos + R @ vc for vc in ibeam_polygon_corners()])
        load_patch.set_xy(corners)

        trail_x.append(pos[0]); trail_y.append(pos[1])
        trail_line.set_data(trail_x, trail_y)

        puller_mask   = hist_puller[step]
        informed_mask = hist_informed[step]
        px, py, lx, ly, ix, iy = [], [], [], [], [], []
        for i, lp in enumerate(local_pos_arr):
            gp = pos + R @ lp
            if informed_mask[i]:
                ix.append(gp[0]); iy.append(gp[1])
            elif puller_mask[i]:
                px.append(gp[0]); py.append(gp[1])
            else:
                lx.append(gp[0]); ly.append(gp[1])

        pullers_plt.set_data(px, py)
        lifters_plt.set_data(lx, ly)
        informed_plt.set_data(ix, iy)
        time_txt.set_text(f't = {step * I2_DT:.1f} s')

        if np.linalg.norm(pos - target) < TARGET_RADIUS:
            print(f"Reached target at t = {step * I2_DT:.2f}s")
            ani.event_source.stop()
        return load_patch, pullers_plt, lifters_plt, informed_plt, trail_line, time_txt

    ani = animation.FuncAnimation(fig, animate, init_func=init,
                                  frames=n_frames, interval=30, blit=True)
    plt.tight_layout()
    if save_gif:
        ani.save(filename, writer='pillow', fps=25)
        print(f"Saved {filename}")
    else:
        plt.show()
    plt.close()
    return ani


# SIDE-BY-SIDE STATIC COMPARISON PLOT

def plot_comparison(n_steps=3000, seed=42):
    """
    Run both implementations and plot their trajectories side by side,
    along with an overlay of the corridor geometry and the I-beam outline
    at key moments (start, corridor entry, corridor exit, end).
    """
    print("Running Impl 1 static sim...")
    load1 = LoadImpl1(start_pos=[-5, -5.0], target=[30.0, 8.0])
    np.random.seed(seed)
    traj1 = [load1.pos.copy()]
    theta1_hist = [load1.theta]
    for _ in range(n_steps):
        load1.update()
        traj1.append(load1.pos.copy())
        theta1_hist.append(load1.theta)
    traj1 = np.array(traj1)

    print("Running Impl 2 static sim...")
    hist_pos2, hist_theta2, _, _, _ = simulate_impl2_with_history(n_steps=n_steps, seed=seed)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
    titles = [
        "Impl 1 — Emergent / penalty-wall",
        "Impl 2 — Paper eqs. (Gillespie-style)",
    ]
    trajs   = [traj1, hist_pos2]
    thetas  = [theta1_hist, hist_theta2]
    colors  = ['saddlebrown', 'steelblue']

    for ax, traj, theta_h, color, title in zip(axes, trajs, thetas, colors, titles):
        ax.set_xlim(-3, 22); ax.set_ylim(-9, 9)
        ax.set_aspect('equal')
        ax.set_facecolor('#f5f0e8')
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("x (cm)")
        ax.grid(True, alpha=0.2)
        ax.set_xlim(BOARD_XMIN, BOARD_XMAX)
        ax.set_ylim(BOARD_YMIN, BOARD_YMAX)
        # Corridor walls
        for wp in get_corridor_patches():
            ax.add_patch(wp)
        ax.axhline( CORRIDOR_HALF_W, color='red', lw=0.8, ls='--', alpha=0.6)
        ax.axhline(-CORRIDOR_HALF_W, color='red', lw=0.8, ls='--', alpha=0.6)
        ax.plot(30.0, 8.0, 'r*', ms=16, zorder=6)

        # Trajectory (time-coloured)
        N = len(traj)
        sc = ax.scatter(traj[::5, 0], traj[::5, 1],
                        c=np.arange(0, N, 5) * I2_DT,
                        cmap='viridis', s=8, zorder=3, alpha=0.7)
        plt.colorbar(sc, ax=ax, label='Time (s)', pad=0.02)
        ax.plot(traj[:, 0], traj[:, 1], color='gray', lw=0.6, alpha=0.4, zorder=2)

        # Draw I-beam outline at 4 key moments
        key_steps = [0,
                     np.argmin(np.abs(traj[:, 0] - CORRIDOR_X_START)),
                     np.argmin(np.abs(traj[:, 0] - CORRIDOR_X_END)),
                     len(traj) - 1]
        beam_alphas = [0.5, 0.7, 0.7, 0.9]
        for ks, ba in zip(key_steps, beam_alphas):
            th = theta_h[ks]
            c2, s2 = np.cos(th), np.sin(th)
            R2 = np.array([[c2, -s2], [s2, c2]])
            corners = np.array([traj[ks] + R2 @ vc for vc in ibeam_polygon_corners()])
            ax.add_patch(Polygon(corners, closed=True,
                                 facecolor=color, edgecolor='black',
                                 alpha=ba, lw=1, zorder=5))

        ax.plot(traj[0, 0], traj[0, 1], 'ko', ms=8, label='Start')
        ax.legend(fontsize=9, loc='lower right')

    axes[0].set_ylabel("y (cm)")
    fig.suptitle("I-beam through narrow corridor — both implementations\n"
                 f"Beam flange width = {2*FLANGE_W:.1f} cm > corridor gap = {2*CORRIDOR_HALF_W:.1f} cm "
                 "(rotation required to pass)",
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig('comparison_ibeam_corridor.png', dpi=140, bbox_inches='tight')
    plt.show()
    print("Saved comparison_ibeam_corridor.png")


# ENTRY POINT — Run everything


if __name__ == '__main__':
    # 1. Side-by-side static comparison (fast, good for a first look)
    plot_comparison(n_steps=3000, seed=42)

    # 2. Animated GIF for Implementation 1
    run_impl1_animation(save_gif=True, filename='impl1_ibeam_corridor.gif',
                        n_frames=500, substeps=5)

    # 3. Animated GIF for Implementation 2 (paper-accurate)
    run_impl2_animation(save_gif=True, filename='impl2_ibeam_corridor.gif',
                        n_steps=3000, stride=5)

    print("\nAll done! Files produced:")
    print("  comparison_ibeam_corridor.png  — static side-by-side trajectory plot")
    print("  impl1_ibeam_corridor.gif       — animated Impl 1")
    print("  impl2_ibeam_corridor.gif       — animated Impl 2")

Running Impl 1 static sim...


D:\ANSYS_TEMP\ipykernel_29028\3268389217.py:254: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  tau_wall += np.cross(r_vec, reaction)
D:\ANSYS_TEMP\ipykernel_29028\3268389217.py:271: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  tau_ants += float(np.cross(r_g, pv))
D:\ANSYS_TEMP\ipykernel_29028\3268389217.py:310: RuntimeWarning: overflow encountered in exp
  if np.random.rand() < I1_K_C * np.exp(energy) * I1_DT:
D:\ANSYS_TEMP\ipykernel_29028\3268389217.py:307: RuntimeWarning: overflow encountered in exp
  if np.random.rand() < I1_K_C * np.exp(-energy) * I1_DT:


Running Impl 2 static sim...


D:\ANSYS_TEMP\ipykernel_29028\3268389217.py:475: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  tau_wall += float(np.cross(r_vec, reaction))
D:\ANSYS_TEMP\ipykernel_29028\3268389217.py:492: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  tau_ants += float(np.cross(r_g, pv))
D:\ANSYS_TEMP\ipykernel_29028\3268389217.py:741: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
D:\ANSYS_TEMP\ipykernel_29028\3268389217.py:254: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  tau_wall += np.cross(r_vec, reaction)
D:\ANSYS_TEMP\ipykernel_29028\3268389217.py:271: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated 

Saved comparison_ibeam_corridor.png
Saved impl1_ibeam_corridor.gif
Running Implementation 2 simulation...


D:\ANSYS_TEMP\ipykernel_29028\3268389217.py:475: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  tau_wall += float(np.cross(r_vec, reaction))
D:\ANSYS_TEMP\ipykernel_29028\3268389217.py:492: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  tau_ants += float(np.cross(r_g, pv))


Saved impl2_ibeam_corridor.gif

All done! Files produced:
  comparison_ibeam_corridor.png  — static side-by-side trajectory plot
  impl1_ibeam_corridor.gif       — animated Impl 1
  impl2_ibeam_corridor.gif       — animated Impl 2
